In [ ]:
import torch
from diffusers import DiffusionPipeline, AutoencoderKL
import os
import yaml

# Load configuration
with open(os.path.expanduser("/work/cvcs2026/stochastic_parrots/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

In [ ]:
MODEL_PATH = config["paths"]["base_model_dir"]
LORA_PATH = config["paths"]["lora_model_dir"]
LORA_WEIGHTS_FILE = config["weights_file"]["lora_weights"]
OUTPUT_DIR = config["paths"]["lora_output_test_dir"]

prompts = [
    "A photo of TOK cat on the moon",                   # Personalizzazione (Soggetto + Contesto)
    "A Renaissance oil painting of TOK cat",           # Personalizzazione (Soggetto + Stile)
    "A photo of TOK cat on a street",
    "A photo of TOK cat with a man",
    "App icon of TOK cat",
    "A TOK cat themed lunchbox",
    "A photo of a golden retriever dog",               # Preservazione (Il modello ricorda altri animali?)
    "A beautiful mountain landscape at sunset",        # Preservazione (C'è "leakage" del gatto nel paesaggio?)
    "A photo of a cat",                                # Prior Preservation (Sa fare ancora gatti generici?)
    "A photo of a beautiful cat"
]

### Load model

In [ ]:
pipe = DiffusionPipeline.from_pretrained(
    MODEL_PATH, 
    torch_dtype=torch.float16, 
    variant="fp16", 
    use_safetensors=True
).to("cuda")  # Carica il modello base in FP16 per risparmiare memoria GPU

pipe.load_lora_weights(
    LORA_PATH,
    weight_name=LORA_WEIGHTS_FILE
)

### Generate images

In [ ]:
for i, prompt in enumerate(prompts):
    print(f"Generating image for prompt {i+1}/{len(prompts)}: '{prompt}'")
    image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
    filename = f"{OUTPUT_DIR}/prompt_{i+1}.png"
    image.save(f"{filename}")